# సేమాంటిక్ కర్నెల్ 

ఈ కోడ్ నమూనాలో, మీరు ఒక సరళ ఏజెంట్ సృష్టించడానికి [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ఫ్రేమ్‌వర్క్‌ను ఉపయోగిస్తారు.

ఈ నమూనా ఉద్దేశ్యం వివిధ ఏజెంటిక్ నమూనాలను అమలు చేసే అదనపు కోడ్ నమూనాల్లో తర్వాత అనుసరించబోయే దశలను ప్రదర్శించడం.


## అవసరమైన Python ప్యాకేజీలు దిగుమతి చేసుకోండి


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## క్లయింట్తో సృష్టించడం

ఈ ఉదాహరణలో, మనం LLM ను యాక్సెస్ చేసేందుకు [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) ఉపయోగిస్తాము.

`ai_model_id` ని `gpt-4o-mini` గా సెట్ చేయబడింది. గిట్హబ్ మోడల్స్ మార్కెట్‌ప్లేస్‌లో అందుబాటులో ఉన్న ఇతర మోడల్‌లకు మార్పులు చేసి వేరే ఫలితాలను పరిశీలించండి.

GitHub Models కోసం అవసరమైన `base_url` కోసం `Azure Inference SDK` ఉపయోగించేందుకు, మనం Semantic Kernel లోని `OpenAIChatCompletion` కనెక్టర్‌ని ఉపయోగిస్తాము. మిగిలిన ఇతర [అవైత్ కనెక్టర్లు](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) కూడా ఉన్నాయి, ఇవి మీకు Semantic Kernel ను ఇతర మోడల్ సరఫరాదారులతో ఉపయోగించుకోవడానికి అనుమతిస్తాయి.


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ఏజెంట్ సృష్టించడం

ఇక్కడ, `TravelAgent` అనే ఏజెంట్‌ను సృష్టిస్తాము.

ఈ ఉదాహరణలో, మేము చాలా ప్రాథమిక సూచనలను ఉపయోగిస్తున్నాము. ఏజెంట్ ప్రవర్తన ఎలా మారుతుందో చూడటానికి ఈ సూచనలను మార్చుకోండి.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## ఏజెంట్లను నడపడం

ప్రస్తుతం మేము `ChatHistory` ని సెటప్ చేసి అందులో `system_message`ని చేర్చడం ద్వారా ఏజెంట్‌ను అమలు చేయవచ్చు. ముందుగా మేము స్థాపించిన `AGENT_INSTRUCTIONS`ని ఉపయోగిస్తాము.

ఇవి సెటప్ చేసిన వెంటనే, మేము `user_inputs` ని సృష్టిస్తాము, ఇది యూజర్ ఏజెంట్‌కు పంపుతున్న విషయాలను సూచిస్తుంది. ఈ ఉదాహరణలో, సందేశం `Plan me a sunny vacation` గా సెట్ చేయబడింది.

ఈ సందేశాన్ని మార్చి ఏజెంట్ ఎలా విభిన్నంగా స్పందిస్తాడో మీరు పరిశీలించవచ్చు.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:
ఈ దస్త్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, స్వయంచాలక అనువాదాల్లో తప్పిదాలు లేదా లోపాలు ఉండవచ్చు. అసలు దస్త్రాన్ని దాని స్వదేశీ భాషలో అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, వృత్తిపరమైన మానవ అనువాదం సిఫార్సు చేయబడుతుంది. ఈ అనువాదం వాడకంలో ఉత్పన్నమయ్యే ఏవైనా అవగాహన లోపాలు లేదా తప్ప فهمలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
